In [3]:
!pip install torch torchvision torchaudio

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 43.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 80.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 70.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.1/684.1 kB 52.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.4 MB/s  0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Attempting uninstall: setuptoolsm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/10 [sympy]]
    Found existing installation: setuptools 82.0.1━━━━━━━━━━━━  2/10 [sympy]
    Uninstalling setuptools-82.0.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/10 [sympy]
      Successfully uninstalled s

In [12]:
import pandas as pd
import time

In [3]:
df_cacao = pd.read_csv(r'../data/data_temp/cacao.csv')

df_x_cacao = df_cacao.iloc[:, 1:]
y_cacao = df_cacao.iloc[:, 0:1]
X_cacao = (df_x_cacao-df_x_cacao.min())/(df_x_cacao.max()-df_x_cacao.min())

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

# 1. Hardware setup for Apple Silicon
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}\n")
import torch
import torch.nn as nn
import math

class SklearnDefaultMLP(nn.Module):
    def __init__(self, in_features, out_features):
     
        super().__init__()
        
        self.hidden = nn.Linear(in_features, 100)
        self.relu = nn.ReLU()        
        self.output = nn.Linear(100, out_features)        
        self._initialize_weights()

    def _initialize_weights(self):
        # sklearn uses Glorot Uniform initialization for BOTH weights and biases.
        for layer in [self.hidden, self.output]:
            fan_in = layer.in_features
            fan_out = layer.out_features
            init_bound = math.sqrt(6.0 / (fan_in + fan_out))
            
            nn.init.uniform_(layer.weight, a=-init_bound, b=init_bound)
            nn.init.uniform_(layer.bias, a=-init_bound, b=init_bound)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)

        return x

# 3. Cross-Validation Setup
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# sklearn MLPClassifier default parameters
max_iter = 8000  # sklearn default (you had 8000)
batch_size = 200  # 'auto' = min(200, n_samples)

input_dim = X_cacao.shape[1]
num_classes = len(np.unique(y_cacao.values))

# List to store the Macro F1 score of each fold
fold_f1_scores = []

print("Starting 5-Fold Cross Validation...\n")
print("-" * 40)

# 4. The Cross-Validation Loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X_cacao.values, y_cacao.values.ravel())):
    print(f"Training Fold {fold + 1}/{n_splits}...")
    
    # Split the data and convert to float32 for MPS compatibility
    X_train = torch.tensor(X_cacao.values[train_idx], dtype=torch.float32)
    X_val = torch.tensor(X_cacao.values[val_idx], dtype=torch.float32)
    
    y_train = torch.tensor(y_cacao.values.ravel()[train_idx], dtype=torch.int64)
    y_val = torch.tensor(y_cacao.values.ravel()[val_idx], dtype=torch.int64)
    
    # Create DataLoaders
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    
    # Keeping shuffle=False to match your sklearn setup requirements
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # CRITICAL: Re-initialize the model, loss, and optimizer for EACH fold
    model = SklearnDefaultMLP(input_dim, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.SGD(model.parameters(),
        lr=0.001,             # sklearn default: learning_rate_init
        momentum=0.9,         # sklearn default: momentum
        nesterov=True,        # sklearn default: nesterovs_momentum
        weight_decay=0.0001   # sklearn default: alpha (L2 regularization)
    )
    
    # --- Training Phase ---
    for epoch in range(max_iter):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
    # --- Evaluation Phase ---
    model.eval()
    all_preds = []
    all_targets = []
    
    # Turn off gradient tracking for evaluation to save memory and speed it up
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            
            outputs = model(inputs)
            
            # The predicted class is the index of the max logit
            _, preds = torch.max(outputs, 1)
            
            # Move predictions back to CPU to use with scikit-learn metrics
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    # Calculate F1 Macro for this fold
    fold_f1 = f1_score(all_targets, all_preds, average='macro')
    fold_f1_scores.append(fold_f1)
    print(f"Fold {fold + 1} F1 Macro: {fold_f1:.4f}")
    print("-" * 40)

# 5. Final Results
print("\nCross-Validation Complete.")
print(f"F1 Macro scores per fold: {[round(score, 4) for score in fold_f1_scores]}")
print(f"Average F1 Macro: {np.mean(fold_f1_scores):.4f} ± {np.std(fold_f1_scores):.4f}")


Using device: mps

Starting 5-Fold Cross Validation...

----------------------------------------
Training Fold 1/5...
Fold 1 F1 Macro: 0.7789
----------------------------------------
Training Fold 2/5...
Fold 2 F1 Macro: 0.7813
----------------------------------------
Training Fold 3/5...
Fold 3 F1 Macro: 0.7813
----------------------------------------
Training Fold 4/5...
Fold 4 F1 Macro: 0.7813
----------------------------------------
Training Fold 5/5...
Fold 5 F1 Macro: 0.7742
----------------------------------------

Cross-Validation Complete.
F1 Macro scores per fold: [0.7789, 0.7813, 0.7813, 0.7813, 0.7742]
Average F1 Macro: 0.7794 ± 0.0027


In [15]:
def k_fold_cross_validation_pytotch(skf, X, y, n_splits, max_iter, batch_size, device):
   
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    max_iter = 8000  # sklearn default (you had 8000)

    input_dim = X.shape[1]
    num_classes = len(np.unique(y.values))

    fold_f1_scores = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X.values, y.values.ravel())):
                
        # Split the data and convert to float32 for MPS compatibility
        X_train = torch.tensor(X.values[train_idx], dtype=torch.float32)
        X_val = torch.tensor(X.values[val_idx], dtype=torch.float32)
        
        y_train = torch.tensor(y.values.ravel()[train_idx], dtype=torch.int64)
        y_val = torch.tensor(y.values.ravel()[val_idx], dtype=torch.int64)
        
        # Create DataLoaders
        train_dataset = TensorDataset(X_train, y_train)
        val_dataset = TensorDataset(X_val, y_val)
        
        # Keeping shuffle=False to match your sklearn setup requirements
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # CRITICAL: Re-initialize the model, loss, and optimizer for EACH fold
        model = SklearnDefaultMLP(input_dim, num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        
        optimizer = torch.optim.SGD(model.parameters(),
            lr=0.001,             # sklearn default: learning_rate_init
            momentum=0.9,         # sklearn default: momentum
            nesterov=True,        # sklearn default: nesterovs_momentum
            weight_decay=0.0001   # sklearn default: alpha (L2 regularization)
        )
        
        # --- Training Phase ---
        for epoch in range(max_iter):
            model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
        # --- Evaluation Phase ---
        model.eval()
        all_preds = []
        all_targets = []
        
        # Turn off gradient tracking for evaluation to save memory and speed it up
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                
                outputs = model(inputs)
                
                # The predicted class is the index of the max logit
                _, preds = torch.max(outputs, 1)
                
                # Move predictions back to CPU to use with scikit-learn metrics
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(labels.numpy())
                
        # Calculate F1 Macro for this fold
        fold_f1 = f1_score(all_targets, all_preds, average='macro')
        fold_f1_scores.append(fold_f1)
    return fold_f1_scores

In [16]:
def cafs(ca,dataset_x,dataset_y, max_iter ,print_logs=False):

  global_data_set = dataset_x
  global_max = 0
  max_iteartion = 0
  result_list_x = []
  result_list_y = []
  result_list_score = []
  result_list_std = []
  num_rows = ca.shape[0]

  if len(dataset_x.columns) < ca.shape[1] :
    num_colums = len(dataset_x.columns)
  else:
    num_colums = ca.shape[1]

  initialTestTraining(result_list_score,result_list_x, result_list_y,dataset_x, dataset_y,result_list_std)
  while max_iteartion < max_iter:

     lst_headers = global_data_set.columns.values.copy()
     max_score = 0.0
     best_std = 0
     mx_data_set = None
     #random.shuffle(lst_headers)
    
     for i in range(0,num_rows):
        lst_headers_to_select = []
        for j in range(0,num_colums ):
          if ca[i][j] == 1 :
              lst_headers_to_select.append(lst_headers[j])
        #with the list of headers to select get sub dat set of col with pandas
        if len(lst_headers_to_select) == 0:
            continue
        df_temp = dataset_x[lst_headers_to_select]
        score = k_fold_cross_validation_pytotch(skf, df_temp, dataset_y, n_splits=5, max_iter=8000, batch_size=200, device=torch.device("mps"))
        if np.mean(score) >= max_score :
            max_score = np.mean(score)
            best_std = np.std(score)
            mx_data_set = df_temp.copy()

     global_data_set = mx_data_set.copy()
     global_max = max_score
     if print_logs:
         print(f"best f1 score= {global_max}, iteration:{max_iteartion}, numbers features selected ={len(global_data_set.columns)},best features selected={', '.join(global_data_set)}" )

     num_colums  = len(global_data_set.columns)
     mx_data_set = None
     max_score = 0
     max_iteartion = max_iteartion  +1
     result_list_x.append(max_iteartion)
     result_list_y.append(len(global_data_set.columns))
     result_list_score.append(global_max)
     result_list_std.append(best_std)
  return result_list_x,result_list_y,result_list_score,result_list_std

def initialTestTraining(score_list, iter_list, feature_list ,X,y,result_list_std):
    score = k_fold_cross_validation_pytotch(skf, X, y, n_splits=5, max_iter=8000, batch_size=200, device=torch.device("mps"))
    score_list.append(np.mean(score))
    feature_list.append(X.shape[1])
    iter_list.append(0)
    result_list_std.append(np.std(score))


In [10]:
covering_array  = np.loadtxt(r'../data/coveringArray.csv', delimiter=",", dtype=int)

In [17]:
start_time = time.time()
iterarions,features,scores,stds = cafs(covering_array,X_cacao,y_cacao,10,True)
#plot_results_for_covering_array(scores,features,iterarions, r'.\\output_images\\cacao_mlp_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

KeyboardInterrupt: 